In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
DATASET_PATH = "/content/drive/MyDrive/pr/Data Set"

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [5]:


img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 636 files belonging to 3 classes.
Using 509 files for training.
Found 636 files belonging to 3 classes.
Using 127 files for validation.
Classes: ['hole', 'lines', 'no defects']


In [6]:
class_names = train_ds.class_names
print("Classes:", class_names)
print("Number of classes:", len(class_names))

Classes: ['hole', 'lines', 'no defects']
Number of classes: 3


In [7]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [9]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
     tf.keras.layers.RandomContrast(0.2),
])

In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),

    data_augmentation,

    tf.keras.layers.Conv2D(32,3,activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64,3,activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128,3,activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(3, activation='softmax')
])

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

In [13]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop, lr_reduce]
)

Epoch 1/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 220s 6s/step - accuracy: 0.5167 - loss: 1.0559 - val_accuracy: 0.5276 - val_loss: 1.0169 - learning_rate: 3.0000e-04
Epoch 2/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 74s 4s/step - accuracy: 0.6189 - loss: 0.9614 - val_accuracy: 0.6614 - val_loss: 0.8991 - learning_rate: 3.0000e-04
Epoch 3/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 72s 4s/step - accuracy: 0.6601 - loss: 0.8144 - val_accuracy: 0.7008 - val_loss: 0.7306 - learning_rate: 3.0000e-04
Epoch 4/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 88s 4s/step - accuracy: 0.6798 - loss: 0.6859 - val_accuracy: 0.7874 - val_loss: 0.5772 - learning_rate: 3.0000e-04
Epoch 5/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 71s 4s/step - accuracy: 0.7230 - loss: 0.5935 - val_accuracy: 0.7795 - val_loss: 0.5247 - learning_rate: 3.0000e-04
Epoch 6/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 73s 4s/step - accuracy: 0.7151 - loss: 0.5725 - val_accuracy: 0.7638 - val_loss: 0.5134 - learning_rate: 3.0000e-04
Epoch 7/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 72s 4s/step - accuracy: 0.6935 - loss

In [14]:
loss, accuracy = model.evaluate(val_ds)

print("Validation Accuracy:", accuracy)

4/4 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.7874 - loss: 0.4702
Validation Accuracy: 0.787401556968689


In [ ]:
import numpy as np

y_pred = model.predict(val_ds)
y_pred_classes = np.argmax(y_pred, axis=1)

4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step


In [15]:
import numpy as np
from sklearn.metrics import classification_report

y_true = np.array([y.numpy() for x, y in val_ds.unbatch()])

y_pred = model.predict(val_ds)
y_pred_classes = np.argmax(y_pred, axis=1)


print(classification_report(y_true, y_pred_classes, zero_division=0))

4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 968ms/step
              precision    recall  f1-score   support

           0       0.50      0.45      0.47        51
           1       0.38      0.38      0.38        47
           2       0.15      0.17      0.16        29

    accuracy                           0.36       127
   macro avg       0.34      0.34      0.34       127
weighted avg       0.37      0.36      0.37       127



In [ ]:
import os

for cls in os.listdir(DATASET_PATH):
    print(cls, len(os.listdir(DATASET_PATH + "/" + cls)))

hole 281
no defects 148
lines 207


In [16]:
model.save("/content/drive/MyDrive/pr/defect_model.keras")

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from google.colab import files

model = tf.keras.models.load_model("/content/drive/MyDrive/pr/defect_model.keras")

In [ ]:
def preprocess(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    return np.expand_dims(img, axis=0)

In [ ]:
def predict():

    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]

    img = preprocess(img_path)

    pred = model.predict(img)

    idx = np.argmax(pred)
    confidence = np.max(pred) * 100

    label = class_names[idx]

    print("\n===== RESULT =====")
    print("Prediction:", label)
    print("Confidence:", f"{confidence:.2f}%")

    if label == "no_defect" or confidence < 60:
        print("GOOD QUALITY (NO DEFECT)")
    else:
        print("DEFECTIVE ITEM")

In [ ]:
predict()

Saving OIP (2).jpg to OIP (2) (1).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step

===== RESULT =====
Prediction: no defect
Confidence: 100.00%
DEFECTIVE ITEM
